In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from losses.dicece_loss import DiceCELoss
from models.nnunet import nnUNet3D
from utils.unets_helper_functions import (
    set_seed,
    train_one_epoch,
    validate_one_epoch,
    save_checkpoint,
    PatchDataset,
    evaluate_full_volume
)

In [3]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:
FOLD = 0

with open(f"../data/splits/fold_{FOLD}/train.txt") as f:
    train_cases = f.read().splitlines()

with open(f"../data/splits/fold_{FOLD}/val.txt") as f:
    val_cases = f.read().splitlines()

print("Train cases:", len(train_cases))
print("Val cases:", len(val_cases))


Train cases: 33
Val cases: 9


- Patch size = 96³
- Patches per case = 12
- Foreground sampling = 0.6
- Batch size = 2 (fallback 1 if OOM)

- Loss = Dice + CE
- Weights = [0.05,1,1,1,1,1,1]
- Optimizer = AdamW (lr 2e-4)
- Scheduler = CosineAnnealing
- Epochs = 80–100


In [5]:
PATCH_SIZE = 96
PATCHES_PER_CASE = 12
FOREGROUND_PROB = 0.6
NUM_WORKERS = 2
train_dataset = PatchDataset(train_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = PATCHES_PER_CASE,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=True)

val_dataset = PatchDataset(val_cases,
                        "../data/processed/imagesTr",
                        "../data/processed/labelsTr",
                        patches_per_case = 1,
                        patch_size = PATCH_SIZE,
                        foreground_prob = FOREGROUND_PROB,
                        augment=False)


train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=NUM_WORKERS,      
    pin_memory=True    
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    num_workers=NUM_WORKERS,    
    pin_memory=True
)

In [6]:
model = nnUNet3D(
    in_channels=1,
    out_channels=7,
    base_filters=24   # or try 32
).to(device)

print("nnU-Net style model Initialized")


nnU-Net style model Initialized


In [7]:
EPOCHS = 100

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

weights = torch.tensor([0.05,1,1,1,1,1,1]).to(device)
loss_fn = DiceCELoss(weight=weights)
scaler = torch.cuda.amp.GradScaler()


C:\Users\dhanu\AppData\Local\Temp\ipykernel_18800\500315588.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "final_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
best_val_loss = float("inf")

best_val_loss = float("inf")

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        loss_fn,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        )
        print("Checkpoint Saved")
    
    scheduler.step()

100%|██████████| 198/198 [11:40<00:00,  3.54s/it]



Epoch 0
Train Loss: 3.0295
Val Loss: 2.4004
Per Class Dice: [0.24096159 0.01826135 0.11111149 0.04302141 0.14297028 0.31391542]
Saved Best Model


100%|██████████| 198/198 [12:40<00:00,  3.84s/it]



Epoch 1
Train Loss: 2.1671
Val Loss: 1.8755
Per Class Dice: [0.62953296 0.41648374 0.3234607  0.32071477 0.39863618 0.6882978 ]
Saved Best Model


100%|██████████| 198/198 [10:33<00:00,  3.20s/it]


...

100%|██████████| 198/198 [10:31<00:00,  3.19s/it]



Epoch 45
Train Loss: 0.8938
Val Loss: 0.8482
Per Class Dice: [0.90281451 0.77314209 0.83109686 0.28989992 0.63293142 0.71342099]


100%|██████████| 198/198 [11:02<00:00,  3.34s/it]



Epoch 46
Train Loss: 0.9211
Val Loss: 0.5382
Per Class Dice: [0.84211541 0.93036295 0.87841804 0.67707598 0.64945167 0.76082578]


 61%|██████    | 120/198 [06:02<02:24,  1.85s/it]

In [8]:
#  RESUME TRAINING 
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "final_nnunet_fold0")

start_epoch = 0
best_val_loss = float("inf")

resume_path = os.path.join(SAVE_DIR, "checkpoint_epoch_40.pth")  

if os.path.exists(resume_path):
    print("Loading checkpoint:", resume_path)

    checkpoint = torch.load(resume_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_epoch = checkpoint["epoch"] + 1
    best_val_loss = checkpoint["best_val_loss"]
    scheduler.last_epoch = start_epoch - 1

    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, starting fresh.")


Loading checkpoint: c:\Users\dhanu\Desktop\mini-project-1\experiments\final_nnunet_fold0\checkpoint_epoch_40.pth


C:\Users\dhanu\AppData\Local\Temp\ipykernel_18800\833551612.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(resume_path, map_location=device)


Resuming from epoch 40


In [9]:
PROJECT_ROOT = os.path.abspath("..")
SAVE_DIR = os.path.join(PROJECT_ROOT, "experiments", "final_nnunet_fold0")

os.makedirs(SAVE_DIR, exist_ok=True)
# best_val_loss = float("inf")

for epoch in range(start_epoch,EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        loss_fn,
        device
    )

    val_loss, val_dice = validate_one_epoch(
        model,
        val_loader,
        loss_fn,
        device
    )

    print(f"\nEpoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Per Class Dice: {val_dice}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, "best_model.pth")
        )
        print("Saved Best Model")

    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        save_checkpoint(
            model,
            optimizer,
            epoch,
            best_val_loss,
            os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch+1}.pth")
        )
        print("Checkpoint Saved")
    
    scheduler.step()

100%|██████████| 198/198 [13:08<00:00,  3.98s/it]



Epoch 40
Train Loss: 0.9182
Val Loss: 1.0648
Per Class Dice: [0.78964309 0.68708734 0.78297799 0.4438036  0.3625842  0.71850369]


100%|██████████| 198/198 [13:53<00:00,  4.21s/it]



Epoch 41
Train Loss: 0.9196
Val Loss: 0.7962
Per Class Dice: [0.93717047 0.73280533 0.79892306 0.27778749 0.5064058  0.85469522]


100%|██████████| 198/198 [12:59<00:00,  3.93s/it]


...


Epoch 97
Train Loss: 0.7020
Val Loss: 0.6172
Per Class Dice: [0.84450788 0.90648358 0.83997999 0.48274405 0.71241564 0.86756296]


100%|██████████| 198/198 [10:23<00:00,  3.15s/it]



Epoch 98
Train Loss: 0.7678
Val Loss: 0.6323
Per Class Dice: [0.94198663 0.86449186 0.88054056 0.62228538 0.60711136 0.92772843]


100%|██████████| 198/198 [10:37<00:00,  3.22s/it]



Epoch 99
Train Loss: 0.7100
Val Loss: 0.4747
Per Class Dice: [0.94630977 0.90762073 0.88696324 0.57360429 0.64115595 0.90347299]
Checkpoint Saved


In [5]:
print("Training Complete")

Training Complete


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nnUNet3D().to(device)

model_path = "../experiments/final_nnunet_fold0/best_model.pth"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded for testing.")


C:\Users\dhanu\AppData\Local\Temp\ipykernel_12472\1838595719.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Model loaded for testing.


with stride = 48

In [6]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.8978452136615971), np.float64(0.8305425319412498), np.float64(0.7538966370920906), np.float64(0.46985142283306586), np.float64(1.943256894785743e-09), np.float64(0.8762388636666358)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9025668513183946), np.float64(0.7725724022380133), np.float64(0.699588477984386), np.float64(0.41779829828997), np.float64(0.11425452641294048), np.float64(0.8680479474337746)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8408707132599949), np.float64(0.8391407429850272), np.float64(0.6774323785275891), np.float64(0.5080541239754864), np.float64(0.24437012347338896), np.float64(0.8840684206667073)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.885770474355598), np.float64(0.7636417778875592), np.float64(0.77953

stride = 48, postprocessing - off

In [6]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.8977033291462012), np.float64(0.8303571429833653), np.float64(0.7538966370920906), np.float64(0.4692425354331033), np.float64(0.1104374912167172), np.float64(0.8759881246421312)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9017134698055349), np.float64(0.7725066009943731), np.float64(0.6994892753840873), np.float64(0.4241250396720305), np.float64(0.20175729509138157), np.float64(0.8672740149582677)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8405094369063958), np.float64(0.8388254721524693), np.float64(0.6788879942004674), np.float64(0.5088580578928186), np.float64(0.2851401594643423), np.float64(0.882130557911972)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.8833870592923508), np.float64(0.7637338099123613), np.float64(0.7795309

stride = 40

In [6]:
mean_dice, std_dice = evaluate_full_volume(
    model,
    val_cases,
    "../data/processed/imagesTr",
    "../data/processed/labelsTr",
    device=device
)

Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_38 Dice: [np.float64(0.9127681842195793), np.float64(0.8321249304181938), np.float64(0.7817444223494029), np.float64(0.5851607092331326), np.float64(0.504537205773367), np.float64(0.8775872265151359)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_05 Dice: [np.float64(0.9071090615358719), np.float64(0.7844546596034645), np.float64(0.7147725000706104), np.float64(0.530783074737043), np.float64(0.4360771961905654), np.float64(0.8735522767136995)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_22 Dice: [np.float64(0.8716353479330766), np.float64(0.8255271766820744), np.float64(0.7171171176834493), np.float64(0.5144499443746607), np.float64(0.23172712589959166), np.float64(0.8834769458761713)]
Unique pred labels: [0 1 2 3 4 5 6]
Unique GT labels: [0 1 2 3 4 5 6]
case_14 Dice: [np.float64(0.910668214683092), np.float64(0.7757730136731246), np.float64(0.757093723

# nnU-Net Style 3D Model — Final Results

Patch-Based Validation Performance

The final model was trained for 100 epochs using nnU-Net style patch-based training with heavy augmentation, Dice+CE loss and balanced foreground sampling.

**Best Validation Epoch
Epoch 44**

Train Loss: 0.9033
Validation Loss: 0.3212

Patch-Based Per-Class Dice (Validation)

- Mandible: 0.9470
- Brainstem: 0.9471
- Spinal Cord: 0.8611
- Parotid Left: 0.8115
- Parotid Right: 0.7386
- Oral Cavity: 0.9382

Observations (Patch Validation)

All organs achieve very high Dice (>0.80) except Parotid-R.
Strong learning of both large and thin structures.
Augmentation + DiceCE loss significantly improved generalization.
Overall, patch-based validation shows excellent learning capacity.

# Evaluating on Full Volume

(Sliding window inference — patch = 96, stride = 40, Gaussian blending + post-processing)

| Organ ID | Organ         | Mean Dice  | Std Dev |
| -------- | ------------- | ---------- | ------- |
| 1        | Bone Mandible | **0.8928** | 0.0275  |
| 2        | Brainstem     | **0.8015** | 0.0210  |
| 3        | Spinal Cord   | **0.7350** | 0.0272  |
| 4        | Parotid Left  | **0.4705** | 0.0663  |
| 5        | Parotid Right | **0.2422** | 0.1312  |
| 6        | Oral Cavity   | **0.8251** | 0.0580  |

# Full Volume Evaluation — Summary

**Strong Organs**

- Mandible (0.89) → Very stable and accurate segmentation
- Brainstem (0.80) → Consistent and reliable predictions
- Oral Cavity (0.82) → Large improvement over earlier models

These organs benefit from:
Larger anatomical context
Clear structural boundaries
Moderate Performance

- Spinal Cord (0.73)

Thin elongated structures are difficult in sliding-window inference.
Despite this, the model achieves strong performance.

Most Challenging Organs

- Parotid Left (0.47)
- Parotid Right (0.24)

Lower performance is expected due to:

Very small organ size
High anatomical variation
Sensitivity to patch sampling
Class imbalance effects

| #     | Model                             | Architecture                   | Base Filters | Patch Size | Patches / Case | FG Sampling | Augmentation           | Batch Size | Loss                   | Optimizer             | Inference                                                 |
| ----- | --------------------------------- | ------------------------------ | ------------ | ---------- | -------------- | ----------- | ---------------------- | ---------- | ---------------------- | --------------------- | --------------------------------------------------------- |
| **1** | **UNet 3D (Baseline)**            | Standard 3D U-Net              | 16           | 80³        | 6              | 0.5         |  Disabled             | 2          | Cross-Entropy          | Adam                  | Sliding window (no Gaussian)                              |
| **2** | **Attention UNet 3D**             | 3D Attention U-Net             | 16           | 80³        | 6              | 0.5         |  Disabled             | 2          | Cross-Entropy          | Adam                  | Sliding window (no Gaussian)                              |
| **3** | **UNet++ 3D**                     | Lightweight Nested U-Net       | 16           | 80³        | 6              | 0.5         |  Disabled             | 2          | Cross-Entropy          | Adam                  | Sliding window (no Gaussian)                              |
| **4** | **nnUNet-style (Initial)**        | Residual 3D U-Net              | 24           | 80³        | 6              | 0.5         |  Disabled             | 2          | Cross-Entropy          | Adam                  | Sliding window (simple averaging)                         |
| **5** | **Final nnUNet Pipeline** | Residual nnUNet-style 3D U-Net | 24           | **96³**    | **12**         | **0.6**     |  TorchIO Augmentation | 2          | **Dice + CE (Hybrid)** | **AdamW + Cosine LR** | **Sliding window + Gaussian weighting + Post-processing** |
